# Conformational Flexibility, Two Ways

**BioPipelines example** — sampling a protein's conformational landscape two ways and comparing them head-to-head. CABSflex runs coarse-grained Monte Carlo and BioEmu emulates the equilibrium ensemble with a deep generative model; EnsembleAnalysis turns each ensemble into a per-residue RMSF profile, and the two are overlaid to see where the methods agree on mobile regions. OpenMM provides the energy-minimised reference structure.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[colab]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines import CABSflex, BioEmu, OpenMM

with Pipeline(project="Setup", job="InstallTools"):
    CABSflex.install()   # NOTE: MODELLER needs a free academic licence key (KEY_MODELLER)
    BioEmu.install()
    OpenMM.install()
    # EnsembleAnalysis needs no install — it runs in the base biopipelines env.

## Cell 3: Flexibility Pipeline

Takes T4 lysozyme (PDB: 2LZM) and compares two flexibility samplers on a common footing:

- **OpenMM** energy-minimises the crystal structure (implicit-solvent AMBER) to give a clean **relaxed reference**.
- **CABSflex** runs coarse-grained Monte Carlo and returns a conformational ensemble (plus its own per-residue RMSF).
- **BioEmu** emulates the equilibrium ensemble directly from sequence with a deep generative model (10 samples).
- **EnsembleAnalysis** converts *each* ensemble into a per-residue RMSF the same way, so the CABSflex and BioEmu profiles are directly comparable rather than coming from two different estimators.
- A **Plot** overlays the two RMSF curves — agreement marks genuinely mobile regions; divergence marks where the methods disagree.

> Computing both RMSF profiles with the same EnsembleAnalysis estimator is the point: it removes the method-specific definition of "fluctuation" as a confound, so the comparison reflects the *sampling*, not the metric.

> **Resource note:** CABSflex and BioEmu are CPU-friendly at this scale; large BioEmu sample counts benefit from a GPU. MODELLER all-atom rebuild in CABSflex needs a free academic licence key set as `KEY_MODELLER`.

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines import PDB, CABSflex, BioEmu, OpenMM, EnsembleAnalysis, Plot

with Pipeline(project="T4Lysozyme", job="Flexibility"):
    Resources(gpu="A100", time="6:00:00", memory="16GB")
    lysozyme = PDB("2LZM", chain="A")

    # Relaxed reference structure (energy minimisation only — not a sampler).
    reference = OpenMM(structures=lysozyme, solvent="implicit-gbn2", max_iterations=1000)

    # Two ensembles: coarse-grained Monte Carlo and a deep-learning emulator.
    cabs = CABSflex(structures=lysozyme, num_models=20, aa_rebuild=False)
    emu  = BioEmu(sequences=lysozyme, num_samples=20)

    # Per-residue RMSF (Cα) from each ensemble, computed by the SAME estimator.
    cabs_rmsf = EnsembleAnalysis(structures=cabs, groups=lysozyme)
    emu_rmsf  = EnsembleAnalysis(structures=emu, groups=lysozyme)

    # Overlay the two per-residue RMSF profiles (tables.residues: id|chain|resi|rmsf|rmsd_mean).
    Plot(Plot.Line(data=[cabs_rmsf.tables.residues, emu_rmsf.tables.residues],
                   x="resi",
                   y="rmsf",
                   labels=["CABSflex", "BioEmu"],
                   title="Per-residue RMSF: CABSflex vs BioEmu (T4 lysozyme)",
                   xlabel="Residue",
                   ylabel="RMSF [Angstrom]"))

    # Ensemble-level comparison: mean RMSF and radius-of-gyration spread per method.
    Plot(Plot.Column(data=[cabs_rmsf.tables.ensemble, emu_rmsf.tables.ensemble],
                     y="mean_rmsf",
                     labels=["CABSflex", "BioEmu"],
                     style="simple_bar",
                     title="Ensemble-mean RMSF by method",
                     ylabel="Mean RMSF [Angstrom]"))
emu_rmsf.tables.ensemble